In [1]:
import json
import os
import requests
from openai import OpenAI
from pydantic import BaseModel, Field

In [2]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [3]:
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"},
]

completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [8]:
print(completion)
print(type(completion))

ChatCompletion(id='chatcmpl-CNgM9zp8J1iRCiBdfTU7JaQRR8jb1', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ldYp1BuS21MLOoXRQChOW0Ni', function=Function(arguments='{"latitude": 59.9139, "longitude": 10.7522}', name='get_weather'), type='function')]))], created=1759760397, model='gpt-5-nano-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=292, prompt_tokens=150, total_tokens=442, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=256, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
<class 'openai.types.chat.chat_completion.ChatCompletion'>


In [10]:
completion.model_dump()
print(completion.model_dump())

{'id': 'chatcmpl-CNgM9zp8J1iRCiBdfTU7JaQRR8jb1', 'choices': [{'finish_reason': 'tool_calls', 'index': 0, 'logprobs': None, 'message': {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': [], 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'call_ldYp1BuS21MLOoXRQChOW0Ni', 'function': {'arguments': '{"latitude": 59.9139, "longitude": 10.7522}', 'name': 'get_weather'}, 'type': 'function'}]}}], 'created': 1759760397, 'model': 'gpt-5-nano-2025-08-07', 'object': 'chat.completion', 'service_tier': 'default', 'system_fingerprint': None, 'usage': {'completion_tokens': 292, 'prompt_tokens': 150, 'total_tokens': 442, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}


In [11]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"}]

In [12]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)


for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    messages.append(completion.choices[0].message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [13]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ldYp1BuS21MLOoXRQChOW0Ni', function=Function(arguments='{"latitude": 59.9139, "longitude": 10.7522}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_ldYp1BuS21MLOoXRQChOW0Ni',
  'content': '{"time": "2025-10-06T14:15", "interval": 900, "temperature_2m": 14.1, "wind_speed_10m": 5.4}'}]

In [14]:
class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )


In [15]:
completion_2 = client.beta.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=WeatherResponse,
)

In [16]:
final_response = completion_2.choices[0].message.parsed
print(final_response.temperature)
print(final_response.response)

14.1
As of 14:15 today in Oslo, it's 14.1°C with a light breeze (wind around 5.4 m/s, ~19 km/h).
